<a href="https://colab.research.google.com/github/AnaSanchez0/mt-esperanto-grupo1/blob/main/fase3_webs_grupo1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Fase 3 – Clasificación de páginas web reales (parte de scraping)

### Paso 1: Páginas elegidas y primera exploración

Clasificar páginas web reales en Esperanto que no sean de la Wikipedia (apartado 4.3.2). Elegimos dos sitios:

* **Kastelo Greziljono**: castillo en Francia, es una casa de cultura del Esperanto. Tiene páginas de alojamiento, de cómo llegar y de turismo.
* **Pasporta Servo**: conjunto de alojamiento gratuito entre esperantistas.

| Página | Clase esperada |
|---|---|
| Greziljono – *Tranoktado* (alojamiento) | hoteles |
| Pasporta Servo – portada | hoteles |
| Greziljono – *Kiel alveni* (cómo llegar) | transporte |
| Greziljono – *Bushoraroj* (horarios de autobús) | transporte |
| Greziljono – *Turisma vizito ĉirkaŭ Baugé* | destinos |
| Greziljono – *Vidindaĵoj ĉirkaŭ Greziljono* (lugares de interés) | destinos |

**Qué hace este paso:**

* **Mira el `robots.txt`** de cada sitio --> Comprueba que nos deja descargar esas páginas (ética del web scraping, tema 3).
* **Descarga cada página una vez** y la enseña por dentro: el título, los encabezados (`h1`, `h2`, `h3`…) y cuántos párrafos (`p`) y elementos de lista (`li`) tiene. Con esto decidimos en el Paso 2 qué parte del texto nos quedamos.
* Entre una página y otra esperamos `PAUSA`, para no saturar los servidores.

In [1]:
# ==== PASO 1: páginas elegidas y primera exploración ====
import time                          # hace pausas
import requests                      # pide páginas a internet
import pandas as pd                  # trabaja con tablas
from bs4 import BeautifulSoup        # lee el HTML de una página (tema 3.3)

URL_REPO = "https://github.com/AnaSanchez0/mt-esperanto-grupo1"   # dirección del repositorio
HEADERS = {"User-Agent": f"MT-practica-grupo1/1.0 (practica de Mineria de textos; contacto: {URL_REPO})"}   # "tarjeta de presentación"
PAUSA = 1                            # segundos entre páginas (para no saturar)

# Páginas de prueba: (nombre, dirección, clase esperada)
PAGINAS = [
    ("Greziljono - Tranoktado",     "https://gresillon.org/rubrique1.html", "hoteles"),
    ("Pasporta Servo - portada",    "https://www.pasportaservo.org/",       "hoteles"),
    ("Greziljono - Kiel alveni",    "https://gresillon.org/rubrique3.html", "transporte"),
    ("Greziljono - Bushoraroj",     "https://gresillon.org/article55.html", "transporte"),
    ("Greziljono - Turisma vizito", "https://gresillon.org/article72.html", "destinos"),
    ("Greziljono - Vidindaĵoj",     "https://gresillon.org/article48.html", "destinos"),
]

def descargar_html(url):
    """Descarga una página y devuelve su HTML o None si falla (el cuaderno no se rompe)."""
    try:
        r = requests.get(url, headers=HEADERS, timeout=15)   # pide la página; espera como máximo 15 s
    except requests.RequestException as e:                  # sin conexión
        print("Error de conexión:", e)
        return None
    if r.status_code != 200:                                # 200 significa "todo bien"; cualquier otro es un error
        print("Código HTTP:", r.status_code)
        return None
    return r.content                                        # el HTML tal cual (pag sin convertir). BeautifulSoup la convierte bien (con ĉ, ĝ, ŭ…)

# --- 1. robots.txt: comprobar que cada web nos deja descargar sus páginas (ética, tema 3.4) ---
for sitio in ["https://gresillon.org", "https://www.pasportaservo.org"]:
    print(f"\n===== robots.txt de {sitio} =====")
    html = descargar_html(sitio + "/robots.txt")            # el robots.txt siempre está en la raíz del sitio
    if html is None:
        print("No tiene robots.txt (por convención, no hay restricciones).")
    else:
        texto = html.decode("utf-8", errors="ignore")       # convierte contenido a texto
        reglas = [l for l in texto.splitlines()
                  if l.lower().startswith(("user-agent", "disallow"))]   # solo las líneas con reglas
        print("\n".join(reglas[:15]))                       # muestra las 15 primeras

# --- 2. Cómo está hecha cada página por dentro --
for nombre, url, clase in PAGINAS:                          # para cada página...
    print(f"\n===== {nombre} ({clase}) =====\n{url}")
    html = descargar_html(url)
    if html is None:
        print("No se ha podido descargar.")                 # si falla (None), pasa a la siguiente página
        continue
    soup = BeautifulSoup(html, "html.parser")               # convierte HTML en algo que se puede recorrer
    titulo = soup.title.text.strip() if soup.title else "(sin título)"   # el título de la pestaña, si no tiene, "(sin título)"
    print("Título:", titulo)
    encabezados = [(h.name, h.text.strip()) for h in soup.find_all(["h1", "h2", "h3", "h4"])]   # títulos de las secciones
    print("Encabezados:", encabezados[:20])                  # los 20 primeros encabezados de la pag.
    print("Párrafos <p>:", len(soup.find_all("p")),          # cuántos párrafos tiene
          "| elementos de lista <li>:", len(soup.find_all("li")))   # y cuántos puntos de lista
    primer_parrafo = next((p.text.strip()[:200] for p in soup.find_all("p")
                           if len(p.text.strip()) > 80), "(ninguno)")   # el primer párrafo de más de 80 caracteres
    print("Primer párrafo largo:", primer_parrafo)
    time.sleep(PAUSA)                                       # espera antes de la siguiente página


===== robots.txt de https://gresillon.org =====
User-agent: *
Disallow: /ecrire/
Disallow: /lib/
Disallow: /oembed.api/
Disallow: /spip.php?action=*
Disallow: /spip.php?page=login*
User-agent: bingbot
User-agent: SemrushBot
Disallow: /
User-agent: BLEXBot
Disallow: /
User-agent: PetalBot
Disallow: /
User-agent: MJ12bot
Disallow: /

===== robots.txt de https://www.pasportaservo.org =====
Código HTTP: 404
No tiene robots.txt (por convención, no hay restricciones).

===== Greziljono - Tranoktado (hoteles) =====
https://gresillon.org/rubrique1.html
Título: Esperanto-programo  - Kulturdomo de Esperanto - Kastelo Greziljono
Encabezados: [('h1', 'Kulturdomo de Esperanto\n\t\t\t\tKastelo Greziljono'), ('h3', 'Lingvoj'), ('h3', 'Rubrikoj'), ('h1', 'Nia Esperanto-programo'), ('h2', 'Disvolviĝo de restado kun kursoj'), ('h2', 'Tranoktado'), ('h2', 'Manĝoj'), ('h2', 'Aliĝo'), ('h2', 'Tarifoj')]
Párrafos <p>: 7 | elementos de lista <li>: 41
Primer párrafo largo: La horaro estas donita kiel ekzempl

### Paso 2: Extracción y limpieza del texto de cada página

En el Paso 1 vimos que las páginas de Greziljono tienen tres partes:

* **Arriba:** el nombre del sitio y los menús (*Lingvoj*, *Rubrikoj*). Son enlaces, no es contenido del tema.
* **En medio:** el contenido de la página. Empieza en su título (por ejemplo, *Kiel atingi Greziljonon…* o *Bushoraroj*).
* **Abajo:** pie de página (dirección, teléfono, *Mapo de la retejo*).

Nos interesa solo el contenido. Por eso, para cada página indicamos:

* `desde`: el título donde empieza el texto que queremos.
* `hasta`: el título de la siguiente sección, donde paramos. En la página del programa, la sección *Tranoktado* (alojamiento) termina donde empieza *Manĝoj* (comidas). En las demás, paramos en el pie de página.

En Pasporta Servo no hay menús ni pie así, y nos quedamos con toda la página.

**Qué hacemos en este paso:**

* Recorrer en orden los párrafos (`p`), puntos de lista (`li`), títulos (`h1`-`h4`) y celdas de tabla (`td`) de la página, y nos quedamos con los que están entre `desde` y `hasta`.
* Si un elemento está dentro de otro que ya hemos cogido (por ejemplo, un párrafo dentro de un punto de lista), lo saltamos para no repetir texto. Y si un elemento envuelve una sección entera (contiene un título), cogemos sus partes una a una, para poder parar justo en el título `hasta`.
* **Problema que encontramos:** al probarlo la 1º vez, el texto de *Tranoktado* salía vacío. En esa página cada sección, con su título, está dentro de un punto de lista (`li`), y el código saltaba todo lo que había dentro de una lista, incluido el título *Tranoktado*. Por eso ahora solo saltamos lo que está dentro de algo que ya hemos cogido.
* **Limpia el texto igual que el dataset** (Fase 1): sin enlaces, sin referencias `[1]`, sin `;` y sin espacios de más.
* **No quitamos las líneas cortas** (`MIN_CHARS`), como sí hicimos en el dataset: estas páginas tienen muchas listas con líneas cortas (horarios, lugares…) que sí hablan del tema. Vamos a clasificar el texto entero de cada página.

In [2]:
# ==== PASO 2: extracción y limpieza del texto de cada página ====
import re                            # buscar trozos de texto

# Qué parte de cada página nos quedamos: (nombre, dirección, clase esperada, desde, hasta)
# desde = empieza el texto; hasta = paramos (None = hasta el final)
PAGINAS = [
    ("Greziljono - Tranoktado",     "https://gresillon.org/rubrique1.html", "hoteles",    "Tranoktado",    "Manĝoj"),
    ("Pasporta Servo - portada",    "https://www.pasportaservo.org/",       "hoteles",    None,            None),
    ("Greziljono - Kiel alveni",    "https://gresillon.org/rubrique3.html", "transporte", "Kiel atingi",   None),
    ("Greziljono - Bushoraroj",     "https://gresillon.org/article55.html", "transporte", "Bushoraroj",    None),
    ("Greziljono - Turisma vizito", "https://gresillon.org/article72.html", "destinos",   "Turisma vizito", None),
    ("Greziljono - Vidindaĵoj",     "https://gresillon.org/article48.html", "destinos",   "Vidindaĵoj",    None),
]
PIE_GREZILJONO = "Mapo de la retejo"                  # pie de página de Greziljono (siempre paramos)
ETIQUETAS = ["h1", "h2", "h3", "h4", "p", "li", "td"]  # títulos, párrafos, puntos de lista y celdas de tabla
TITULOS = ["h1", "h2", "h3", "h4"]

def limpiar(texto):
    """Limpia el texto igual que en el dataset (Fase 1)."""
    texto = re.sub(r"<[^>]+>", " ", texto)            # etiquetas HTML: "<", lo que sea, ">"
    texto = re.sub(r"http\S+|www\.\S+", " ", texto)   # enlaces
    texto = re.sub(r"\[\d+\]", " ", texto)            # referencias como [1]
    texto = texto.replace(";", ",")                   # ";" prohibido (es el separador del CSV)
    texto = re.sub(r"\s+", " ", texto)                # varios espacios o saltos de línea -> un espacio
    return texto.strip()

def extraer_texto(html, desde=None, hasta=None):
    """Saca el texto de la página entre el título 'desde' y el título 'hasta'."""
    soup = BeautifulSoup(html, "html.parser")         # convierte HTML en algo que se puede recorrer
    lineas = []                                       # guardamos aqui el texto que nos quedamos
    cogidos = set()                                   # elementos ya cogidos (para no repetir su texto)
    dentro = desde is None                            # sin 'desde'-> empezamos desde el principio
    for el in soup.find_all(ETIQUETAS):               # en orden todos los elementos con texto
        texto = limpiar(el.get_text(" "))             # texto con un espacio entre partes, y limpio
        if not texto:                                 # vacío -> se salta
            continue
        if el.name in TITULOS:                        # queremos saber si empieza o termina lo que queremos en el título
            if not dentro and el.name in ("h1", "h2") and desde in texto:
                dentro = True                         # En desde -> a partir de aquí cogemos el texto
            elif dentro and hasta and hasta in texto:
                break                                 # sección siguiente ->paramos
        if not dentro:                                # aún no llegamos a 'desde'
            continue
        if PIE_GREZILJONO in texto:                   # pie de página -> paramos
            break
        if el.name not in TITULOS and el.find(TITULOS):   # envuelve sección entera: cogemos sus partes una a una
            continue
        if any(id(padre) in cogidos for padre in el.parents):   # dentro de algo ya cogido (no repetir)
            continue
        cogidos.add(id(el))                           # lo añadimos a cogido
        lineas.append(texto)
    return " ".join(lineas)                           # texto de la página en una sola línea

# --- Descargar, extraer y limpiar cada página ---
filas = []
for nombre, url, clase, desde, hasta in PAGINAS:
    html = descargar_html(url)                        # función Paso 1
    texto = extraer_texto(html, desde, hasta) if html else ""   # si no se ha podido descargar, texto vacío
    filas.append({"pagina": nombre, "url": url, "clase_esperada": clase, "text": texto})
    time.sleep(PAUSA)                                 # espera antes de la siguiente página

webs = pd.DataFrame(filas)                            # tabla con una fila por página
webs["n_palabras"] = webs["text"].str.split().str.len()   # cuántas palabras tiene cada texto
print(webs[["pagina", "clase_esperada", "n_palabras"]])

# --- Comprobación: principio y final de cada texto ---
for _, fila in webs.iterrows():
    print(f"\n===== {fila['pagina']} ({fila['n_palabras']} palabras) =====")
    print("PRINCIPIO:", fila["text"][:300])           # los primeros 300 caracteres (elegimos 300 para comprobar)
    print("FINAL:    ", fila["text"][-300:])          # los últimos 300 caracteres (elegimos 300 para comprobar)

                        pagina clase_esperada  n_palabras
0      Greziljono - Tranoktado        hoteles         238
1     Pasporta Servo - portada        hoteles         902
2     Greziljono - Kiel alveni     transporte         696
3      Greziljono - Bushoraroj     transporte         874
4  Greziljono - Turisma vizito       destinos        1265
5      Greziljono - Vidindaĵoj       destinos         894

===== Greziljono - Tranoktado (238 palabras) =====
PRINCIPIO: Tranoktado 25 gastoĉambroj kun ĉ. 60 litoj disponeblas en la 1a kaj 2a etaĝoj de la kastelo kaj en la kromdomoj : 6 ĉambroj kun 1 lito : n-roj 10, 12, 13, 14, 16, 21 9 ĉambroj kun 2 litoj (aŭ 1 duopa lito) : n-roj 5, 6, 15, 17, 18, 19, 22, 23, 37 PMR 4 ĉambroj kun 3 litoj (unuopaj aŭ duopa) : n-ro 7
FINAL:     s sanitarejo kun 2 lavkuvoj, 2 duŝejoj kaj 2 necesejoj. Interreto haveblas en la biblioteko per kabla konekto. Sendrata Wifi-konekto eblas se ne ĉeestas gastoj kun e lektro-magneta h iper- s entemo EHS . La fiksa telefo

### Paso 3: Copia de seguridad en GitHub (plan B) y guardado

Se pide que el cuaderno funcione aunque falle cualquier web. Por eso, igual que en la Fase 1, guardamos los textos de las páginas en nuestro repositorio de GitHub (`textos_web.csv`).

**Qué hace este paso:**

* **Carga la copia de GitHub** (si existe).
* **Si una página no se ha podido descargar** (texto vacío), usa su texto de la copia de GitHub. En la columna `origen` aparece dónde sale cada texto: `web`, `copia de GitHub` o `sin texto` (si fallan la web y GitHub a la vez).
* **Guarda los textos en `textos_web.csv`** (UTF-8, separador `;`), pero solo si **todas** las páginas se han descargado bien de la web. Así la copia de GitHub siempre está completa y actualizada.
* **Prueba del plan B:** hemos simulado que *Bushoraroj* no se puede descargar (borrando su texto). El cuaderno usa su texto de la copia de GitHub (las mismas 874 palabras) y no actualiza `textos_web.csv`. Está comentado, por si se quiere repetir.

Este fichero es también **lo que usamos para clasificar las páginas**: tiene el texto de cada una y la clase que esperamos, para comparar después con lo que diga el modelo.

In [3]:
# ==== PASO 3: copia de seguridad en GitHub (plan B) y guardado ====
URL_RAW = "https://raw.githubusercontent.com/AnaSanchez0/mt-esperanto-grupo1/main"   # dirección "en bruto" (pandas la puede leer)
URL_WEBS = f"{URL_RAW}/textos_web.csv"     # copia de los textos en GitHub
FICHERO_WEBS = "textos_web.csv"            # fichero que guardamos

def cargar_csv(url):
    """Carga un CSV (separador ';') desde una dirección. Devuelve None si falla."""
    try:
        datos = pd.read_csv(url, sep=";", encoding="utf-8") # lee CSV desde la dirección
        print(f"Cargado {url} ({len(datos)} filas)")        # avisa de que lo ha cargado y cuántas filas tiene
        return datos                                        # devuelve la tabla
    except Exception as e:                 # sin conexión, 404 (aún no existe), fichero mal formado...
        print(f"No se ha podido cargar {url}: {e}")
        return None

respaldo = cargar_csv(URL_WEBS)            # la copia de GitHub (None si no existe o GitHub no responde)
# PRUEBA plan B: quitar "#" de la línea de abajo si queremos que Bushoraroj no se pueda descargar
# webs.loc[3, "text"] = ""

# --- Plan B: si una página falla, usamos su texto de la copia ---
webs["origen"] = "web"                     # de momento, todos los textos vienen de la web
for i, fila in webs.iterrows():            # se repite para cada página
    if fila["text"] == "":                 # la página no se ha podido descargar
        copia = respaldo[respaldo["url"] == fila["url"]] if respaldo is not None else pd.DataFrame()   # su fila en la copia
        if len(copia) > 0:                 # está en la copia: la usamos
            webs.loc[i, "text"] = copia["text"].iloc[0]
            webs.loc[i, "origen"] = "copia de GitHub"
        else:                              # tampoco está en la copia (fallan la web y GitHub)
            webs.loc[i, "origen"] = "sin texto"
            print(f"{fila['pagina']}: no hay texto (ni de web ni de GitHub). No se clasificará.")

webs["n_palabras"] = webs["text"].str.split().str.len()   # vuelve a contar las palabras
print(webs[["pagina", "clase_esperada", "origen", "n_palabras"]])

# --- Guardar: si todas las páginas vienen de web (así la copia siempre está completa) ---
if (webs["origen"] == "web").all():
    webs[["pagina", "url", "clase_esperada", "text"]].to_csv(FICHERO_WEBS, sep=";", index=False, encoding="utf-8")
    print(f"\nGuardado {FICHERO_WEBS} ({len(webs)} páginas). En el plan B, este fichero se sube a GitHub.")
else:
    print(f"\nAlguna página no viene de la web. No se actualiza {FICHERO_WEBS}.")

Cargado https://raw.githubusercontent.com/AnaSanchez0/mt-esperanto-grupo1/main/textos_web.csv (6 filas)
                        pagina clase_esperada origen  n_palabras
0      Greziljono - Tranoktado        hoteles    web         238
1     Pasporta Servo - portada        hoteles    web         902
2     Greziljono - Kiel alveni     transporte    web         696
3      Greziljono - Bushoraroj     transporte    web         874
4  Greziljono - Turisma vizito       destinos    web        1265
5      Greziljono - Vidindaĵoj       destinos    web         894

Guardado textos_web.csv (6 páginas). En el plan B, este fichero se sube a GitHub.


### Paso 4: Extraer el texto de cualquier página web

En los Pasos 1 a 3 trabajamos con nuestras 6 páginas. Como ya conocíamos su estructura, para cada una dijimos dónde empieza el texto y dónde acaba (`desde` y `hasta`). Además, su texto está guardado en GitHub (`textos_web.csv`) por si alguna no responde.

Pero el modelo tiene que servir también para **páginas nuevas** que no hemos visto anteriormente. Para esas páginas:
- no sabemos dónde está el texto, así que no podemos usar `desde` y `hasta`;
- no hay copia en GitHub, así que si la web falla no podemos usar el plan B del Paso 3.

**Qué hace `texto_de_url(url)`:**
1. **Descarga** la página con `descargar_html` (Paso 1). Si no se puede descargar, avisa y devuelve `None`. Así el cuaderno no se rompe y el clasificador sabe que esa página no se puede clasificar.
2. **Busca el contenido principal.** Si la página lo marca con `<main>` o `<article>`, solo mira ahí. Si no, mira el `<body>` entero.
3. **Recoge el texto** de títulos (`h1`-`h4`), párrafos (`p`), puntos de lista (`li`) y celdas de tabla (`td`), con `find_all` (tema 3.3).
4. **Se salta lo que no es contenido:** lo que está dentro de menús (`nav`), cabecera (`header`), pie (`footer`), barras laterales (`aside`), formularios y código (`script`, `style`). Para eso mira los padres de cada elemento con `.parents`. Igual que en el Paso 2, no coge dos veces el mismo texto.
5. **Limpia el texto** con la función `limpiar()` del Paso 2, la misma que el dataset: quita etiquetas HTML, enlaces y referencias tipo `[1]`, cambia `;` por `,` y quita espacios de más.
6. **Avisa** si la página tiene muy poco texto (menos de 50 palabras) o si no parece estar en Esperanto (menos de un 5 % de palabras muy comunes como *la*, *kaj*, *estas*, *de*). Estos avisos no paran el cuaderno. Solo ayudan a entender un resultado raro.

**Limitación.** Algunas webs no marcan sus menús con `<nav>`, sino con listas normales. En esas webs se puede colar algo de texto del menú. Por eso, para nuestras 6 páginas seguimos usando la extracción a medida del Paso 2, que es más precisa. Esta función general es para las páginas nuevas.

La función usa el mismo User-Agent y la misma descarga del Paso 1. Antes de usar una página nueva hay que mirar a mano su `robots.txt` (dirección de la web + `/robots.txt`) para comprobar que permite el acceso.

**Prueba:** la última celda prueba la función con dos casos:
* *Turisma vizito* (una de nuestras páginas): salen 1261 palabras, frente a las 1265 del Paso 2. Solo falta el título de la página. Empieza en el contenido y termina en los créditos, sin menú ni pie de página.
* Una página que no existe: da error 404, sale el aviso y la función devuelve `None`. El cuaderno no se rompe.

In [4]:
# ==== PASO 4: extraer el texto de cualquier página web ====
ZONAS_FUERA = {"nav", "header", "footer", "aside", "script", "style", "form"}  # partes que no son contenido
ETIQUETAS_TEXTO = ["h1", "h2", "h3", "h4", "p", "li", "td"]                     # dónde suele estar el texto
MIN_PALABRAS_WEB = 50        # menos de esto: muy poco texto para fiarse del resultado
PALABRAS_EO = {"la", "kaj", "estas", "de", "en", "al", "por", "kun", "ne", "ĉe"}  # palabras muy comunes en Esperanto

def texto_de_url(url):
    """Devuelve el texto limpio de una página web, o None si no se puede descargar."""
    html = descargar_html(url)                      # la función del Paso 1 (devuelve None si falla)
    if html is None:
        print(f"AVISO: no se pudo descargar {url}.")
        print("       Esta página no está en la copia de GitHub, así que no se puede clasificar.")
        return None

    soup = BeautifulSoup(html, "html.parser")
    # Si la página marca su contenido principal (<main> o <article>), usamos solo eso
    zona = soup.find("main") or soup.find("article") or soup.body or soup

    trozos = []
    cogidos = set()
    for el in zona.find_all(ETIQUETAS_TEXTO):
        nombres_padres = {p.name for p in el.parents}
        if nombres_padres & ZONAS_FUERA:            # está dentro de un menú, cabecera o pie → fuera
            continue
        if any(id(p) in cogidos for p in el.parents):  # ya cogido dentro de otro elemento → no repetir
            continue
        t = el.get_text(" ").strip()
        if t:
            trozos.append(t)
            cogidos.add(id(el))

    # Limpieza (la misma idea que en el dataset)
    texto = limpiar(" ".join(trozos))      # misma limpieza que Paso 2 y en el dataset

    # Avisos (no paran el cuaderno, ayudan a entender un resultado raro)
    palabras = texto.lower().split()
    if len(palabras) < MIN_PALABRAS_WEB:
        print(f"AVISO: {url} tiene muy poco texto ({len(palabras)} palabras).")
    if palabras:
        prop_eo = sum(p in PALABRAS_EO for p in palabras) / len(palabras)
        if prop_eo < 0.05:
            print(f"AVISO: {url} no parece estar en Esperanto ({prop_eo:.0%} de palabras típicas).")

    return texto

In [5]:
# --- Prueba Paso 4 ---
# 1) Una de nuestras páginas: tiene que salir casi lo mismo que en el Paso 2 (1265 palabras)
t = texto_de_url("https://gresillon.org/article72.html")
print(len(t.split()), "palabras")
print("PRINCIPIO:", t[:300])
print("FINAL:    ", t[-300:])

# 2) Una página que no existe: tiene que salir el AVISO y None, sin error
print(texto_de_url("https://gresillon.org/esta-pagina-no-existe.html"))

1261 palabras
PRINCIPIO: Tiu semajno celas logi esp-istojn kiuj deziras ferii en turisma regiono kaj emas malkovri la imponan kulturan heredaĵon de la franca regiono inter du riveroj : Luaro (france la Loire ) kaj Loir ( le Loir ) La konstruistoj el la pasinteco en tiu areo profitis el donacoj de la naturo, nome ŝtono kalka
FINAL:     de la valo Luaro ekloĝis la francaj reĝoj, nome Karlo la 7a en Chinon. | Tiun turisman semajnon en januaro 2011 elpensis-redaktis Jannick Huet-Schumann , konsilis-helpis Michèle Jagueneau kaj Geneviève Gourrichon , enretigis-bildigis Bert Schumann. Pli kompletan liston de la Luar-kasteloj vidu tie .
Código HTTP: 404
AVISO: no se pudo descargar https://gresillon.org/esta-pagina-no-existe.html.
       Esta página no está en la copia de GitHub, así que no se puede clasificar.
None
